# rl-tunix SWE-bench Warm Pools — Demo

This notebook walks through using **Agent Sandbox warm pools** to run
[SWE-bench](https://www.swebench.com/) tasks the way the
[rl-tunix](https://github.com/google/tunix) RL/eval pipeline does: pre-warm pools
of sandboxes per task image, claim one sandbox per task, run a command inside it,
and tear down.

It exercises the three warm-pool strategies (`none`, `naive`, `sliding`) from
this example's `rl_simple_scripts/strategies.py` against a live cluster, using
**real** `R2E-Gym/SWE-Bench-Verified` images.

**Prerequisites** (see `README.md`): a cluster with the Agent Sandbox controller
+ extensions installed, `pip install -r rl_simple_scripts/requirements.txt`, and a
kubeconfig pointed at the cluster. SWE-bench images are multi-GB, so keep the task
count small and allow time for the first pull.

In [ ]:
# Install deps (uncomment on first run)
# %pip install -r requirements.txt

## 1. Configuration
Kept small for a demo. Set `NODE_SELECTOR_*` to pin to a pool, `RUNTIME_CLASS=gvisor` for isolation.

In [ ]:
import logging, os
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")

NAMESPACE = "rl-tunix-swebench"
TASKS_LIMIT = 2                 # tasks to pull from the dataset
MAX_WARMPOOL_SIZE = 1          # replicas cap per image pool (tiny for the demo)
WINDOW_SIZE = 1               # sliding-window width
SANDBOX_READY_TIMEOUT = 900   # multi-GB image pulls are slow on first use

# Optional cluster pinning / isolation:
NODE_SELECTOR = None          # e.g. {"cloud.google.com/gke-nodepool": "standard-pool"}
RUNTIME_CLASS = None          # e.g. "gvisor"
IMAGE_PULL_SECRET = None      # e.g. "dockerhub-pro"

## 2. Connect to the cluster

In [ ]:
from kubernetes import client, config
from k8s_agent_sandbox import SandboxClient

import strategies, warmpool as wp

config.load_kube_config()
custom_api = client.CustomObjectsApi()
core_api = client.CoreV1Api()
sandbox_client = SandboxClient()

# Make sure the namespace exists.
try:
    core_api.create_namespace(client.V1Namespace(metadata=client.V1ObjectMeta(name=NAMESPACE)))
    print("created namespace", NAMESPACE)
except client.ApiException as e:
    print("namespace:", "exists" if e.status == 409 else e.status)

## 3. Inspect real SWE-bench task entries
Each entry carries a real `docker_image` (the repo checked out at a commit).

In [ ]:
from datasets import load_dataset

ds = load_dataset("R2E-Gym/SWE-Bench-Verified", split="test")
entries = [
    {"instance_id": r["instance_id"], "docker_image": r["docker_image"], "repo": r.get("repo", "")}
    for r in ds
][:TASKS_LIMIT]

for e in entries:
    print(e["instance_id"], "->", e["docker_image"])

## 4. Define how each task is processed
Claim a pre-warmed sandbox from the image's pool, exec a probe command (router-free, via `kubectl exec`), then terminate.

In [ ]:
results = []

mgr = strategies.WarmPoolManager(
    custom_api, NAMESPACE,
    node_selector=NODE_SELECTOR,
    image_pull_secret=IMAGE_PULL_SECRET,
    runtime_class=RUNTIME_CLASS,
    ready_timeout=SANDBOX_READY_TIMEOUT,
)

PROBE = ["bash", "-lc", "echo READY $(hostname); git -C /testbed log -1 --oneline 2>/dev/null || ls /"]

def process(task, pool):
    sb = sandbox_client.create_sandbox(warmpool=pool, namespace=NAMESPACE,
                                       sandbox_ready_timeout=SANDBOX_READY_TIMEOUT)
    try:
        pod = sb.get_pod_name()
        out = wp.exec_in_pod(core_api, pod, NAMESPACE, PROBE)
        print(f"[{task['instance_id']}] pod={pod}: {out.strip()}")
        results.append({"instance_id": task["instance_id"], "pod": pod, "output": out.strip()})
    finally:
        sb.terminate()

## 5. Strategy: `naive`
Pre-warm a pool for **every** unique image up front, run all tasks, tear down.

In [ ]:
results.clear()
strategies.run_naive(entries, mgr, process, max_warmpool_size=MAX_WARMPOOL_SIZE)
results

## 6. Strategy: `sliding`
Sort by image and keep only `WINDOW_SIZE` pools warm at a time, rolling forward
as each image's tasks finish. Best for large, image-diverse batches.

In [ ]:
results.clear()
strategies.run_sliding(entries, mgr, process, window_size=WINDOW_SIZE, max_warmpool_size=MAX_WARMPOOL_SIZE)
results

## 7. Strategy: `none`
No pre-warming — a size-1 pool is provisioned on demand per task. Highest
per-task latency, lowest idle cost.

In [ ]:
results.clear()
strategies.run_none(entries, mgr, process)
results

## 8. Cleanup
The strategies tear down their own pools; this removes anything left over.

In [ ]:
# Cleanup. The strategies tear down their own pools; this removes any leftovers.
# Note: SandboxWarmPool/Claim/Template are in the *extensions* group, while the
# core Sandbox CRD is in the agents.x-k8s.io group.
TARGETS = [
    ("extensions.agents.x-k8s.io", "sandboxwarmpools"),
    ("extensions.agents.x-k8s.io", "sandboxclaims"),
    ("extensions.agents.x-k8s.io", "sandboxtemplates"),
    ("agents.x-k8s.io", "sandboxes"),
]
for group, plural in TARGETS:
    objs = custom_api.list_namespaced_custom_object(group, "v1beta1", NAMESPACE, plural).get("items", [])
    for o in objs:
        name = o["metadata"]["name"]
        try:
            custom_api.delete_namespaced_custom_object(
                group, "v1beta1", NAMESPACE, plural, name,
                body=client.V1DeleteOptions(grace_period_seconds=0))
            print("deleted", plural, name)
        except client.ApiException as e:
            if e.status != 404:
                raise
print("done")